# 03b · Prediction only: saved LightGBM model → submission files

Use this when `lightgbm-submission-2.ipynb` finished training but stopped during test prediction (the
Kaggle kernel ran out of RAM on India, the largest test country, with the training data still in memory).

It does **not** retrain. It loads the saved model and runs only the prediction and file-writing part:

1. The settings cell and the **feature code are copied** from `lightgbm-submission-2.ipynb`, so the 61
   features are computed exactly as when the model was trained. The notebook stops if the model's feature
   list differs. Three functions are replaced by faster versions that give **bit-identical values**
   (checked on 2.3M texts and 600K pairs): the word-rarity overlaps and the shared street words / numbers
   are computed with sparse matrices instead of a Python loop per pair, and accent folding skips plain
   ASCII text. Together about 1.5x faster features and 20% faster record loading.
2. Each test country is predicted in `PRED_PARTS` groups of S1: only the records of one group are loaded at
   a time. The features that compare a candidate across all S1 of the country, and the one-owner rule,
   still use the whole country, so the result is identical to a single pass.
3. A country whose probabilities were already saved by the failed run (France) is reused after checking
   that its pairs are exactly the bucket pairs.
4. Matches: keep p ≥ t with one owner per S2/S3 (t = the cross-validation cutoff saved in
   `metric_f05.csv`, 0.70), then write and check `matching_results.tsv` and `candidate_pairs.tsv`.

**Attach on Kaggle:** the competition data, 01's output (`processed/`), 02 full's output (the buckets) and
the failed run's output (`submission/model/lgbm_matcher.txt`, `metric_f05.csv`,
`test_probabilities/France.parquet`; `oof_pairs.parquet` optional, for the comparison table).
Accelerator: none needed (CPU). Estimated time: about 1.5-2 hours (France is reused; India and US are
about 54M pairs; France's 10M pairs took 18 minutes).

In [ ]:
import collections
import csv
import gc
import glob
import importlib.util
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
import unicodedata
from pathlib import Path

import itertools

import numpy as np
import scipy.sparse as sp
import pandas as pd
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

for pkg in ("lightgbm", "rapidfuzz", "anyascii"):   # MIT, MIT, ISC
    if importlib.util.find_spec(pkg) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
import lightgbm as lgb
from rapidfuzz import fuzz
from anyascii import anyascii
from rapidfuzz.distance import JaroWinkler

CANDIDATES_PER_S1 = int(os.environ.get("CANDIDATES_PER_S1", 30))          # top e5 candidates judged per S1 (30 beat 20 on the test bed)
USE_EXTRA_CANDIDATES = os.environ.get("USE_EXTRA_CANDIDATES", "true") == "true"  # also judge 02's extra candidates (if present)
TRAIN_S1_PER_COUNTRY = int(os.environ.get("TRAIN_S1_PER_COUNTRY", 200_000))  # train S1 used for learning, per country
N_FOLDS = 3
SEED = 42
LGBM_DEVICE = os.environ.get("LGBM_DEVICE", "cpu")                        # auto | cpu | gpu | cuda
CROSS_COUNTRY_CHECK = True                                                  # train on one country, score the other
PAIRS_PER_CHUNK = 2_000_000                                                 # pairs featurized at a time (memory)
FEATURE_SET = os.environ.get("FEATURE_SET", "all")                          # all | base (the 44 of notebook 03)
COUNTRY_THRESHOLDS = {}   # optional stricter cutoff per test country, e.g. {"France": 0.9}; others use the tuned t
T0 = time.time()


def search(patterns, roots):
    """First existing file matching any glob pattern under any root (shortest path wins); skips __MACOSX."""
    for root in roots:
        if root and os.path.isdir(root):
            hits = [h for pat in patterns for h in glob.glob(os.path.join(root, pat), recursive=True) if "__MACOSX" not in h]
            if hits:
                return Path(sorted(hits, key=len)[0])
    return None


KAGGLE = os.path.isdir("/kaggle/working")
LOCAL_DATA = str(Path.cwd().parent / "Data" / "student_resource")
if os.environ.get("BUCKET_DIR"):
    BUCKET_DIR = Path(os.environ["BUCKET_DIR"])
else:
    hit = search(["**/buckets/train_*.parquet", "**/buckets/test_*.parquet"],
                 ["embeddings_full", "/kaggle/working/embeddings_full", "/kaggle/input"])
    assert hit is not None, "no buckets found: run 02_full_e5_buckets.ipynb (or attach its output) or set BUCKET_DIR"
    BUCKET_DIR = hit.parent
MANIFEST = json.loads((BUCKET_DIR.parent / "manifest.json").read_text(encoding="utf-8")) \
    if (BUCKET_DIR.parent / "manifest.json").exists() else {}
proc = search(["train_s1_*.parquet", "**/train_s1_*.parquet"],
              [os.environ.get("PROCESSED_DIR", ""), MANIFEST.get("settings", {}).get("processed_dir", ""), "processed",
               "/kaggle/working/processed", "/kaggle/input"])
assert proc is not None, "processed/ not found: attach 01's output or set PROCESSED_DIR"
PROCESSED_DIR = proc.parent
DATA_ROOTS = [os.environ.get("DATA_ROOT", ""), "/kaggle/input", LOCAL_DATA]
GT_FILE = search(["**/train_ground_truth.tsv"], DATA_ROOTS)
TEST_S1_FILE = search(["**/test_source1.tsv"], DATA_ROOTS)
assert TEST_S1_FILE is not None, "competition data not found: attach it or set DATA_ROOT"
TEST_DIR = TEST_S1_FILE.parent
VALIDATOR = Path(os.environ["VALIDATOR"]) if os.environ.get("VALIDATOR") else \
    search(["**/validate_submission.py"], ["/kaggle/input", LOCAL_DATA])
OUT_DIR = Path(os.environ.get("SUBMISSION_DIR") or ("/kaggle/working/submission" if KAGGLE else "submission"))
for sub in ("output", "model", "test_probabilities"):
    (OUT_DIR / sub).mkdir(parents=True, exist_ok=True)
print(f"buckets: {BUCKET_DIR}\nprocessed: {PROCESSED_DIR}\nground truth: {GT_FILE}\ntest data: {TEST_DIR}\n"
      f"validator: {VALIDATOR}\noutput: {OUT_DIR}")
BUCKETS = sorted(p.stem for p in BUCKET_DIR.glob("*_*.parquet"))
TRAIN_COUNTRIES = [b.split("_", 1)[1] for b in BUCKETS if b.startswith("train_")]
TEST_COUNTRIES = [b.split("_", 1)[1] for b in BUCKETS if b.startswith("test_")]
print(f"train buckets: {TRAIN_COUNTRIES} | test buckets: {TEST_COUNTRIES}")
assert TEST_COUNTRIES, "no test buckets found (attach 02 full's output)"


def lgbm_device():
    """GPU if this LightGBM build supports it (tried on a tiny dataset), else CPU."""
    if LGBM_DEVICE != "auto":
        return LGBM_DEVICE
    if shutil.which("nvidia-smi") is None:
        return "cpu"
    x = np.random.default_rng(0).random((2000, 4))
    for device in ("cuda", "gpu"):
        try:
            lgb.train({"objective": "binary", "device_type": device, "verbose": -1},
                      lgb.Dataset(x, (x[:, 0] > 0.5).astype(int)), num_boost_round=2)
            return device
        except Exception:
            pass
    print("this LightGBM build has no GPU support, training on the CPU (the 'Fatal' lines above come from that test)")
    return "cpu"


DEVICE = lgbm_device()
PARAMS = {"objective": "binary", "learning_rate": 0.1, "num_leaves": 63, "min_data_in_leaf": 100,
          "feature_fraction": 0.8, "bagging_fraction": 0.8, "bagging_freq": 1, "lambda_l2": 1.0,
          "max_bin": 255 if DEVICE == "cpu" else 63, "device_type": DEVICE, "seed": SEED, "verbose": -1}
print(f"LightGBM {lgb.__version__} on {DEVICE}; {CANDIDATES_PER_S1} candidates per S1; "
      f"{TRAIN_S1_PER_COUNTRY:,} train S1 per country; {N_FOLDS} folds")

## Feature code (from `lightgbm-submission-2.ipynb`; `weighted_overlap`, `set_overlap` and `fold_text` replaced by faster versions with identical values)

In [ ]:
try:
    from rapidfuzz.process import cpdist   # rapidfuzz >= 3.6: many pairs at once on all CPU cores
except ImportError:
    cpdist = None

STR_COLS = ["business_name", "business_address", "name_norm", "addr_norm", "core_name", "name_lat", "legal_form",
            "domain_stem", "phonetic_code", "street_tokens", "house_no", "city", "state_code"]
FLAG_COLS = ["has_honorific", "empty_addr", "landmark_flag"]
TAGS = ["tag_native_script", "tag_web_name", "tag_empty_address", "tag_honorific", "tag_landmark"]
# a name has a native script when it has a character outside Latin, Latin-1/Extended and general punctuation
_LATIN_RANGES = [(0x00, 0x7F), (0xA0, 0x24F), (0x1E00, 0x1EFF), (0x2000, 0x206F), (0x20A0, 0x20CF)]
RE_NATIVE = re.compile("[^" + "".join(re.escape(chr(a)) + "-" + re.escape(chr(b)) for a, b in _LATIN_RANGES) + "]")
RE_HANDLE = re.compile(r"^\s*[@#]\S+\s*$")
RE_NUM = re.compile(r"\d+")
RE_POSTCODE = re.compile(r"(?<!\d)\d{5,6}(?!\d)")
RE_NOT_ALNUM = re.compile(r"[^0-9a-z]+")
FEATURES = ["rank", "score", "cosine", "s1_best_cosine", "gap_to_best_score", "gap_to_next_score", "s1_top5_cosine",
            "cand_n_s1", "cand_rank", "cand_margin", "name_ratio", "name_token_set", "name_token_sort", "name_partial",
            "core_ratio", "core_jaro_winkler", "latin_ratio", "latin_token_set", "raw_name_token_set", "compact_ratio",
            "compact_partial", "phonetic_same", "legal_form_same", "s1_name_len", "cand_name_len", "addr_ratio",
            "addr_token_set", "raw_addr_token_set", "raw_addr_partial", "street_jaccard", "street_cover",
            "numbers_jaccard", "numbers_cover", "postcode_same", "house_no_same", "city_same", "state_same",
            "cand_addr_len", "cand_is_s3", "cand_native_script", "cand_web_name", "cand_empty_address",
            "cand_honorific", "cand_landmark"]
BASE_FEATURES = list(FEATURES)
FEATURES = FEATURES + ["name_idf_jaccard", "name_idf_cover", "name_idf_miss_s1", "name_idf_miss_cand", "addr_idf_jaccard",
                       "addr_idf_cover", "addr_idf_miss_s1", "addr_idf_miss_cand", "street_name_token_set",
                       "street_name_ratio", "house_number_same", "s1_name_freq", "cand_name_freq"]
SOURCE_FEATURES = ["e5_rank", "rev_rank", "addr_key", "name_rank"]              # written by 02 with extra sources
FEATURES = FEATURES + SOURCE_FEATURES
if FEATURE_SET == "base":
    FEATURES = BASE_FEATURES
_ACCENTS = {}
for _cp in range(0xC0, 0x250):
    _base = unicodedata.normalize("NFKD", chr(_cp))[0]
    if _base.isascii() and _base != chr(_cp):
        _ACCENTS[_cp] = _base
FOLD = str.maketrans(_ACCENTS)
RE_WORD = re.compile(r"\w+")
STREET_TYPES = frozenset("rue avenue av ave boulevard bd blvd boul chemin che place pl route rte allee impasse imp quai "
                         "cours crs square faubourg fbg road rd street st lane ln drive dr way court ct circle cir "
                         "highway hwy parkway pkwy terrace marg".split())
STREET_STOP = STREET_TYPES | frozenset("r ch all sq bis ter de du des la le les l d of the and et en sur suite ste unit "
                                       "apt apartment building bldg floor fl po box door no nr n number h hno flat shop "
                                       "plot near opp".split())


def fold_text(s):
    """Lower case with Latin accents removed (other scripts unchanged)."""
    return s.lower() if s.isascii() else s.lower().translate(FOLD)


def words(s):
    """Folded word tokens; numbers without leading zeros ('0058' -> '58')."""
    return [(w.lstrip("0") or "0") if w.isdigit() else w for w in RE_WORD.findall(fold_text(s))]


def street_of(address):
    """(street name, house number): the first comma part with a number or a street word, minus numbers, street
    words and stop words. '15 bis Rue de Barbieux, Roubaix' -> ('barbieux', '15')."""
    house = ""
    for part in address.split(","):
        w = words(part)
        nums = [x for x in w if x.isdigit()]
        if nums and not house:
            house = nums[0]
        if nums or any(x in STREET_TYPES for x in w):
            name = " ".join(x for x in w if not x.isdigit() and x not in STREET_STOP and len(x) > 1)
            if name:
                return name, house
    return "", house


def country_stats(split, country):
    """Word rarity (IDF) in the names and addresses of all S1 of one split and country, and how many S1 share each name."""
    s1 = pd.read_parquet(PROCESSED_DIR / f"{split}_s1_{country}.parquet", columns=["core_name", "business_address"])
    names = [frozenset(words(x)) for x in s1["core_name"].fillna("").astype(str)]
    addrs = [frozenset(words(x)) for x in s1["business_address"].fillna("").astype(str)]
    n = len(s1)

    def idf(sets):
        df = collections.Counter(t for s in sets for t in s)
        return {t: math.log((n + 1) / (c + 1)) + 1.0 for t, c in df.items()}

    return {"name_idf": idf(names), "addr_idf": idf(addrs), "unseen_idf": math.log(n + 1) + 1.0, "n": n,
            "name_freq": collections.Counter(" ".join(sorted(s)) for s in names)}


def _indicator_rows(a, b):
    """Sparse 0/1 rows (one per pair) for two aligned arrays of sets, over their shared vocabulary."""
    codes_a, sets_a = pd.factorize(np.asarray(a, dtype=object))
    codes_b, sets_b = pd.factorize(np.asarray(b, dtype=object))
    len_a = np.fromiter(map(len, sets_a), np.int64, len(sets_a))
    len_b = np.fromiter(map(len, sets_b), np.int64, len(sets_b))
    n_a = int(len_a.sum())
    flat = np.fromiter(itertools.chain.from_iterable(itertools.chain(sets_a, sets_b)), object, n_a + int(len_b.sum()))
    word_ids, vocab = pd.factorize(flat)
    n_words = max(len(vocab), 1)
    ma = sp.csr_matrix((np.ones(n_a), word_ids[:n_a], np.r_[0, np.cumsum(len_a)]), shape=(len(sets_a), n_words))[codes_a]
    mb = sp.csr_matrix((np.ones(len(word_ids) - n_a), word_ids[n_a:], np.r_[0, np.cumsum(len_b)]),
                       shape=(len(sets_b), n_words))[codes_b]
    return ma, mb, vocab


def weighted_overlap(a, b, idf, unseen):
    """IDF-weighted Jaccard, share of the S1's weight found in the candidate, and the rarest unmatched word on each
    side, for aligned arrays of word sets (NaN when either side is empty). Sparse matrices: one row of word
    indicators per pair, weights summed / maximized per row in C (same values as a loop over the pairs)."""
    out = np.full((4, len(a)), np.nan, np.float32)
    if not len(a):
        return out
    ma, mb, vocab = _indicator_rows(a, b)
    w = np.fromiter((idf.get(t, unseen) for t in vocab), np.float64, len(vocab))
    inter = ma.multiply(mb).tocsr()
    wi, wx, wy = inter @ w, ma @ w, mb @ w
    ok = (np.diff(ma.indptr) > 0) & (np.diff(mb.indptr) > 0)
    only_a, only_b = (ma - inter).tocsr(), (mb - inter).tocsr()
    only_a.eliminate_zeros()
    only_b.eliminate_zeros()
    with np.errstate(divide="ignore", invalid="ignore"):
        out[0, ok] = (wi / (wx + wy - wi))[ok]
        out[1, ok] = (wi / wx)[ok]
    out[2, ok] = only_a.multiply(w).tocsr().max(axis=1).toarray().ravel()[ok]
    out[3, ok] = only_b.multiply(w).tocsr().max(axis=1).toarray().ravel()[ok]
    return out


def load_records(split, country, ids):
    """Fields of the given entity IDs from processed/<split>_s{1,2,3}_<country>.parquet, plus per-record helpers."""
    wanted = pa.array(sorted(ids), pa.large_string())
    frames = []
    for src in ("s1", "s2", "s3"):
        f = PROCESSED_DIR / f"{split}_{src}_{country}.parquet"
        if not f.exists():
            continue
        names = set(pq.read_schema(f).names)
        t = pq.read_table(f, columns=["entity_id"] + [c for c in STR_COLS + FLAG_COLS if c in names])
        t = t.filter(pc.is_in(t["entity_id"].cast(pa.large_string()), value_set=wanted))
        frames.append(t.to_pandas())
    ent = pd.concat(frames, ignore_index=True).drop_duplicates("entity_id").set_index("entity_id")
    for c in STR_COLS:
        ent[c] = ent[c].fillna("").astype(str) if c in ent else ""
    for c in FLAG_COLS:
        ent[c] = pd.to_numeric(ent[c], errors="coerce").fillna(0).astype(np.int8) if c in ent else 0
    name = ent["business_name"].tolist()
    ent["tag_native_script"] = [RE_NATIVE.search(x) is not None for x in name]
    # native-script names/addresses: every comparison uses the original text transliterated (01's is garbled)
    native_name = ent["tag_native_script"].to_numpy()
    if native_name.any():
        lat = [" ".join(anyascii(x).lower().split()) for x in ent.loc[native_name, "business_name"]]
        for c in ("business_name", "name_norm", "core_name", "name_lat"):
            ent.loc[native_name, c] = lat
    native_addr = np.array([RE_NATIVE.search(x) is not None for x in ent["business_address"]])
    if native_addr.any():
        lat = [anyascii(x) for x in ent.loc[native_addr, "business_address"]]
        ent.loc[native_addr, "business_address"] = lat
        ent.loc[native_addr, "addr_norm"] = [" ".join(x.lower().replace(",", " ").split()) for x in lat]
    ent["tag_web_name"] = (ent["domain_stem"] != "").to_numpy() | np.array([RE_HANDLE.match(x) is not None for x in name])
    ent["tag_empty_address"] = (ent["empty_addr"] == 1) | (ent["addr_norm"] == "")
    ent["tag_honorific"] = ent["has_honorific"] == 1
    ent["tag_landmark"] = ent["landmark_flag"] == 1
    ent["raw_name"] = ent["business_name"].str.lower()
    ent["raw_addr"] = ent["business_address"].str.lower()
    ent["compact"] = [RE_NOT_ALNUM.sub("", (d or l or n).lower())
                      for d, l, n in zip(ent["domain_stem"], ent["name_lat"], ent["name_norm"])]
    ent["name_words"] = [frozenset(words(x)) for x in ent["core_name"]]
    ent["addr_words"] = [frozenset(words(x)) for x in ent["business_address"]]
    ent["numbers"] = [frozenset(x for x in w if x.isdigit()) for w in ent["addr_words"]]   # '0058' == '58'
    streets = [street_of(x) for x in ent["business_address"]]
    ent["street_name"] = [s for s, _ in streets]
    ent["house_number"] = [h for _, h in streets]
    ent["name_key"] = [" ".join(sorted(w)) for w in ent["name_words"]]
    ent["postcode"] = [(RE_POSTCODE.findall(a) or [""])[-1] for a in ent["raw_addr"]]
    ent["street_set"] = [frozenset(s.split()) for s in ent["street_tokens"]]
    return ent.drop(columns=["business_name", "business_address", "street_tokens", "domain_stem"] + FLAG_COLS)


def read_buckets(split, country):
    """Bucket rows of one (split, country), grouped by S1 in rank order, with candidate-side stats: the top
    CANDIDATES_PER_S1 e5 candidates, plus 02's extra candidates when present (and USE_EXTRA_CANDIDATES)."""
    path = BUCKET_DIR / f"{split}_{country}.parquet"
    if "e5_rank" in pq.read_schema(path).names:                      # 02 with extra sources: rank = position after filtering
        b = pq.read_table(path).to_pandas()
        e5 = b["e5_rank"].to_numpy()
        keep = (e5 < CANDIDATES_PER_S1) | (np.isnan(e5) & USE_EXTRA_CANDIDATES)
        b = b[keep].reset_index(drop=True)
        s1 = b["s1_entity_id"].to_numpy()
        starts = np.r_[0, np.flatnonzero(s1[1:] != s1[:-1]) + 1]
        b["rank"] = (np.arange(len(b)) - np.repeat(starts, np.diff(np.r_[starts, len(b)]))).astype(np.int16)
    else:
        b = pq.read_table(path, filters=[("rank", "<", CANDIDATES_PER_S1)]).to_pandas()
    s1 = b["s1_entity_id"].to_numpy()
    assert int((s1[1:] != s1[:-1]).sum()) + 1 == b["s1_entity_id"].nunique(), "bucket rows are not grouped by S1"
    # how the candidate scores against every S1 of the country that has it in its bucket
    codes, uniq = pd.factorize(b["candidate_entity_id"])
    score = b["score"].to_numpy(np.float32)
    order = np.lexsort((-score, codes))
    sc, ss = codes[order], score[order]
    start = np.r_[True, sc[1:] != sc[:-1]]
    r_sorted = np.arange(len(sc)) - np.maximum.accumulate(np.where(start, np.arange(len(sc)), 0))
    cand_rank = np.empty(len(b), np.float32)
    cand_rank[order] = r_sorted
    best = np.full(len(uniq), np.nan, np.float32)
    best[sc[start]] = ss[start]
    second = np.full(len(uniq), np.nan, np.float32)
    second[sc[r_sorted == 1]] = ss[r_sorted == 1]
    b["cand_n_s1"] = np.bincount(codes, minlength=len(uniq)).astype(np.float32)[codes]
    b["cand_rank"] = cand_rank
    b["cand_margin"] = np.where(cand_rank == 0, score - second[codes], score - best[codes]).astype(np.float32)
    return b


def chunk_bounds(b):
    """(start, end) row ranges of about PAIRS_PER_CHUNK pairs that never split an S1's bucket."""
    s1 = b["s1_entity_id"].to_numpy()
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    idx = np.searchsorted(starts, np.arange(PAIRS_PER_CHUNK, len(b), PAIRS_PER_CHUNK))
    cuts = np.unique(np.r_[0, starts[idx[idx < len(starts)]], len(b)])
    return list(zip(cuts[:-1], cuts[1:]))


def sim(scorer, a, b):
    """rapidfuzz similarity of aligned string arrays; NaN where either side is empty."""
    if cpdist is not None:
        out = cpdist(list(a), list(b), scorer=scorer, workers=-1).astype(np.float32)
    else:
        out = np.array([scorer(x, y) for x, y in zip(a, b)], np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def same(a, b):
    """1 when equal, 0 when different, NaN when either side is empty."""
    out = (a == b).astype(np.float32)
    out[(a == "") | (b == "")] = np.nan
    return out


def set_overlap(a, b):
    """Jaccard overlap and share of a's items found in b, for aligned arrays of frozensets (NaN when both empty)."""
    jac = np.full(len(a), np.nan, np.float32)
    cover = np.full(len(a), np.nan, np.float32)
    if not len(a):
        return jac, cover
    ma, mb, _ = _indicator_rows(a, b)
    na, nb = np.diff(ma.indptr), np.diff(mb.indptr)
    inter = np.asarray(ma.multiply(mb).sum(axis=1)).ravel()
    both, one = (na > 0) & (nb > 0), (na > 0) ^ (nb > 0)
    jac[both] = (inter / np.maximum(na + nb - inter, 1))[both]
    cover[both] = (inter / np.maximum(na, 1))[both]
    jac[one] = cover[one] = 0.0
    return jac, cover


def make_features(b, ent, stats):
    """float32 feature matrix (columns FEATURES) for the bucket rows b (whole S1 buckets, in rank order)."""
    qpos = ent.index.get_indexer(b["s1_entity_id"])
    cpos = ent.index.get_indexer(b["candidate_entity_id"])
    missing = int((qpos < 0).sum() + (cpos < 0).sum())
    assert missing == 0, f"{missing} bucket records are missing from processed/"

    def pair(col):
        v = ent[col].to_numpy()
        return v[qpos], v[cpos]

    f = {}
    by_s1 = b.groupby("s1_entity_id", sort=False)
    f["rank"] = b["rank"].to_numpy(np.float32)
    f["score"], f["cosine"] = b["score"].to_numpy(np.float32), b["cosine"].to_numpy(np.float32)
    f["s1_best_cosine"] = by_s1["cosine"].transform("max").to_numpy(np.float32)
    f["gap_to_best_score"] = (by_s1["score"].transform("max") - b["score"]).to_numpy(np.float32)
    f["gap_to_next_score"] = (b["score"] - by_s1["score"].shift(-1)).to_numpy(np.float32)
    f["s1_top5_cosine"] = b["s1_entity_id"].map(b[b["rank"] < 5].groupby("s1_entity_id")["cosine"].mean()).to_numpy(np.float32)
    for c in ("cand_n_s1", "cand_rank", "cand_margin"):
        f[c] = b[c].to_numpy(np.float32)
    x, y = pair("name_norm")
    f["name_ratio"], f["name_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    f["name_token_sort"], f["name_partial"] = sim(fuzz.token_sort_ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["s1_name_len"] = np.fromiter((len(v) for v in x), np.float32, len(x))
    f["cand_name_len"] = np.fromiter((len(v) for v in y), np.float32, len(y))
    x, y = pair("core_name")
    f["core_ratio"], f["core_jaro_winkler"] = sim(fuzz.ratio, x, y), sim(JaroWinkler.normalized_similarity, x, y)
    x, y = pair("name_lat")
    f["latin_ratio"], f["latin_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    x, y = pair("raw_name")
    f["raw_name_token_set"] = sim(fuzz.token_set_ratio, x, y)
    x, y = pair("compact")
    f["compact_ratio"], f["compact_partial"] = sim(fuzz.ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["phonetic_same"] = same(*pair("phonetic_code"))
    f["legal_form_same"] = same(*pair("legal_form"))
    x, y = pair("addr_norm")
    f["addr_ratio"], f["addr_token_set"] = sim(fuzz.ratio, x, y), sim(fuzz.token_set_ratio, x, y)
    x, y = pair("raw_addr")
    f["raw_addr_token_set"], f["raw_addr_partial"] = sim(fuzz.token_set_ratio, x, y), sim(fuzz.partial_ratio, x, y)
    f["cand_addr_len"] = np.fromiter((len(v) for v in y), np.float32, len(y))
    f["street_jaccard"], f["street_cover"] = set_overlap(*pair("street_set"))
    f["numbers_jaccard"], f["numbers_cover"] = set_overlap(*pair("numbers"))
    f["postcode_same"] = same(*pair("postcode"))
    f["house_no_same"] = same(*pair("house_no"))
    f["city_same"] = same(*pair("city"))
    f["state_same"] = same(*pair("state_code"))
    f["cand_is_s3"] = (b["candidate_source"].to_numpy() == "S3").astype(np.float32)
    for tag in TAGS:
        f[f"cand_{tag[4:]}"] = pair(tag)[1].astype(np.float32)
    if FEATURE_SET == "base":
        return np.column_stack([f[c] for c in FEATURES]).astype(np.float32)
    for kind in ("name", "addr"):
        w = weighted_overlap(*pair(f"{kind}_words"), stats[f"{kind}_idf"], stats["unseen_idf"])
        for j, c in enumerate(("jaccard", "cover", "miss_s1", "miss_cand")):
            f[f"{kind}_idf_{c}"] = w[j]
    x, y = pair("street_name")
    f["street_name_token_set"], f["street_name_ratio"] = sim(fuzz.token_set_ratio, x, y), sim(fuzz.ratio, x, y)
    f["house_number_same"] = same(*pair("house_number"))
    x, y = pair("name_key")
    freq, per = stats["name_freq"], 100_000 / stats["n"]             # S1 with this name per 100,000 S1 of the country
    f["s1_name_freq"] = np.log1p(per * np.fromiter((freq.get(v, 0) for v in x), np.float32, len(x)))
    f["cand_name_freq"] = np.log1p(per * np.fromiter((freq.get(v, 0) for v in y), np.float32, len(y)))
    for c in SOURCE_FEATURES:                                        # empty when 02 ran without extra sources
        f[c] = b[c].to_numpy(np.float32) if c in b.columns else np.full(len(b), np.nan, np.float32)
    return np.column_stack([f[c] for c in FEATURES]).astype(np.float32)


def featurize(b, ent, stats):
    """make_features over chunks of whole S1 buckets."""
    return np.vstack([make_features(b.iloc[s:e], ent, stats) for s, e in chunk_bounds(b)])


print(f"{len(FEATURES)} features")

## Saved model, cutoff and prediction settings
* `MODEL_FILE`: found under `/kaggle/input` (`**/model/lgbm_matcher.txt`) unless set.
* `T_BEST`: the cutoff of the row "LightGBM p ≥ t + one owner" in `metric_f05.csv` next to the model
  (0.70 in the failed run), unless `T_BEST` is set.
* `PRED_PARTS`: S1 groups per country. Raise it (e.g. 16) if memory still runs out. Fewer groups are
  faster (records shared by several groups are prepared once per group; 8 groups prepare each record ~3x,
  4 groups ~2.2x) but need more memory (India's largest group: ~8 GB of records at 8 groups, ~12 GB at 4).
* `REUSE_PROBABILITIES`: reuse the probabilities the failed run saved (France).

In [ ]:
PRED_PARTS = int(os.environ.get("PRED_PARTS", 8))                      # S1 groups per test country (memory)
REUSE_PROBABILITIES = os.environ.get("REUSE_PROBABILITIES", "true") == "true"
MODEL_FILE = Path(os.environ["MODEL_FILE"]) if os.environ.get("MODEL_FILE") else \
    search(["**/model/lgbm_matcher.txt", "**/lgbm_matcher.txt"], ["/kaggle/input", "submission", "results"])
assert MODEL_FILE is not None and MODEL_FILE.exists(), "attach the failed run's output (model/lgbm_matcher.txt) or set MODEL_FILE"
FINAL = lgb.Booster(model_file=str(MODEL_FILE))
assert FINAL.feature_name() == FEATURES, "the model was trained with other features than this notebook computes"
metric_file = MODEL_FILE.parent / "metric_f05.csv"
if os.environ.get("T_BEST"):
    T_BEST = float(os.environ["T_BEST"])
elif metric_file.exists():
    m = pd.read_csv(metric_file)
    T_BEST = float(m.loc[m["method"].str.startswith("LightGBM p ≥ t + one owner"), "t"].iloc[0])
else:
    T_BEST = 0.70
PREV_PROB_DIR = MODEL_FILE.parent.parent / "test_probabilities"
print(f"model: {MODEL_FILE} ({FINAL.num_trees()} trees, {FINAL.num_feature()} features) | t = {T_BEST} | "
      f"country thresholds: {COUNTRY_THRESHOLDS or 'none'} | {PRED_PARTS} S1 groups per country")
print(f"saved probabilities to reuse: {sorted(p.name for p in PREV_PROB_DIR.glob('*.parquet')) if REUSE_PROBABILITIES and PREV_PROB_DIR.exists() else 'none'}")

## Predict the test buckets
Per country: read all its bucket pairs (with the candidate-side statistics over the whole country), then
for each S1 group load only that group's records, compute the features in chunks and predict. The
probability of every pair is saved in `test_probabilities/<country>.parquet`, as in the training notebook.

In [ ]:
def part_bounds(b, parts):
    """(start, end) row ranges of about len(b) / parts rows that never split an S1's bucket."""
    s1 = b["s1_entity_id"].to_numpy()
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    targets = np.linspace(0, len(b), parts + 1)[1:-1]
    cuts = starts[np.minimum(np.searchsorted(starts, targets), len(starts) - 1)]
    cuts = np.unique(np.r_[0, cuts, len(b)])
    return list(zip(cuts[:-1], cuts[1:]))


def same_pairs(old, b):
    """True when a saved probability table holds exactly the bucket pairs of b, in the same order."""
    return len(old) == len(b) and (old["s1_entity_id"].to_numpy() == b["s1_entity_id"].to_numpy()).all() \
        and (old["candidate_entity_id"].to_numpy() == b["candidate_entity_id"].to_numpy()).all()


for country in TEST_COUNTRIES:
    t0 = time.time()
    out_file = OUT_DIR / "test_probabilities" / f"{country}.parquet"
    b = read_buckets("test", country)
    prev = PREV_PROB_DIR / f"{country}.parquet"
    if REUSE_PROBABILITIES and prev.exists() and prev.resolve() != out_file.resolve():
        old = pd.read_parquet(prev)
        if same_pairs(old, b):
            shutil.copyfile(prev, out_file)
            print(f"test {country}: {len(b):,} pairs, probabilities reused from {prev}")
            del b, old
            gc.collect()
            continue
        print(f"test {country}: saved probabilities do not match the buckets, predicting again")
        del old
    stats = country_stats("test", country)
    prob = np.empty(len(b), np.float32)
    for k, (lo, hi) in enumerate(part_bounds(b, PRED_PARTS)):
        part = b.iloc[lo:hi]
        ent = load_records("test", country, set(part["s1_entity_id"]) | set(part["candidate_entity_id"]))
        for s, e in chunk_bounds(part):
            prob[lo + s:lo + e] = FINAL.predict(make_features(part.iloc[s:e], ent, stats))
        del ent, part
        gc.collect()
        print(f"  {country} group {k + 1}/{PRED_PARTS}: rows {lo:,}-{hi:,}, {time.time() - t0:.0f}s", flush=True)
    pd.DataFrame({"s1_entity_id": b["s1_entity_id"], "candidate_entity_id": b["candidate_entity_id"],
                  "rank": b["rank"], "p": prob}).to_parquet(out_file, index=False)
    print(f"test {country}: {b['s1_entity_id'].nunique():,} S1, {len(b):,} pairs predicted in {time.time() - t0:.0f}s")
    del b, prob, stats
    gc.collect()

## Matches: p ≥ t and one owner per S2/S3
The same rule as the training notebook, applied to the saved probabilities of each country. Check France
here: compare its "S1 with a predicted match" and "matches per matched S1" with India and the US and
with the cross-validation rates below them (version 1: France 95.8% / 3.67, US 94.1% / 3.52, India
92.1% / 3.23). For a stricter France cutoff, set `COUNTRY_THRESHOLDS` in the settings cell and re-run
from this cell, or use `04_rethreshold.ipynb` on this notebook's output.

In [ ]:
def join_by_s1(s1, ids):
    """{s1: 'id,id,...'} for rows grouped by S1 (order kept, duplicates removed)."""
    if not len(s1):
        return {}
    starts = np.flatnonzero(np.r_[True, s1[1:] != s1[:-1]])
    ends = np.r_[starts[1:], len(s1)]
    return {s1[a]: ",".join(dict.fromkeys(ids[a:e])) for a, e in zip(starts, ends)}


def one_owner(df, p):
    """True for the rows where this S1 gives the candidate its highest p among all S1 in df."""
    order = np.argsort(-p, kind="stable")
    first = ~pd.Series(df["candidate_entity_id"].to_numpy()[order]).duplicated().to_numpy()
    owner = np.zeros(len(df), bool)
    owner[order[first]] = True
    return owner


CANDIDATES, MATCHES, TEST_STATS = {}, {}, []
for country in TEST_COUNTRIES:
    d = pd.read_parquet(OUT_DIR / "test_probabilities" / f"{country}.parquet")
    prob = d["p"].to_numpy()
    t_country = COUNTRY_THRESHOLDS.get(country, T_BEST)
    keep = (prob >= t_country) & one_owner(d, prob)
    CANDIDATES.update(join_by_s1(d["s1_entity_id"].to_numpy(), d["candidate_entity_id"].to_numpy()))
    kept = pd.DataFrame({"s1": d["s1_entity_id"].to_numpy()[keep], "cand": d["candidate_entity_id"].to_numpy()[keep],
                         "p": prob[keep]}).sort_values(["s1", "p"], ascending=[True, False])
    MATCHES.update(join_by_s1(kept["s1"].to_numpy(), kept["cand"].to_numpy()))
    n_s1 = d["s1_entity_id"].nunique()
    TEST_STATS.append({"country": country, "t": t_country, "S1 with buckets": n_s1, "pairs scored": len(d),
                       "S1 with a predicted match": round(kept["s1"].nunique() / n_s1, 4),
                       "matches per matched S1": round(len(kept) / max(kept["s1"].nunique(), 1), 2)})
    del d, prob, keep, kept
    gc.collect()
display(pd.DataFrame(TEST_STATS))
oof_file = MODEL_FILE.parent / "oof_pairs.parquet"
if oof_file.exists():
    o = pd.read_parquet(oof_file, columns=["country", "s1_entity_id", "kept"])
    print("cross-validation rates (train), for comparison:")
    display(o.groupby("country").apply(lambda g: pd.Series({
        "S1 with a predicted match (cv)": g.loc[g["kept"], "s1_entity_id"].nunique() / g["s1_entity_id"].nunique(),
        "matches per matched S1 (cv)": g["kept"].sum() / max(g.loc[g["kept"], "s1_entity_id"].nunique(), 1)}),
        include_groups=False).round(3))
    del o

## Write the submission files
Every S1 in `test_source1.tsv` gets exactly one row, in file order, with an empty list when it
has no candidates or no match. The files are tab-separated, UTF-8 and use `\n` line endings,
with the headers from the problem statement.

In [ ]:
def read_ids(path):
    """First-column IDs of a TSV, read exactly as validate_submission.py reads them (file order, no duplicates)."""
    with open(path, encoding="utf-8") as f:
        next(f, None)
        return list(dict.fromkeys(line.split("\t", 1)[0].strip() for line in f if line.strip()))


def write_tsv(path, header, values, s1_ids):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\t".join(header) + "\n")
        for s in s1_ids:
            f.write(f"{s}\t{values.get(s, '')}\n")


TEST_S1 = read_ids(TEST_S1_FILE)
MATCHING_FILE = OUT_DIR / "output" / "matching_results.tsv"
CANDIDATE_FILE = OUT_DIR / "output" / "candidate_pairs.tsv"
write_tsv(MATCHING_FILE, ["source1_entity_id", "matched_entity_ids"], MATCHES, TEST_S1)
write_tsv(CANDIDATE_FILE, ["source1_entity_id", "candidate_entity_ids"], CANDIDATES, TEST_S1)
not_in_test = set(CANDIDATES) - set(TEST_S1)
print(f"{len(TEST_S1):,} test S1 written | with candidates: {sum(s in CANDIDATES for s in TEST_S1):,} | "
      f"with matches: {sum(s in MATCHES for s in TEST_S1):,} | bucket S1 not in test_source1.tsv: {len(not_in_test)}")
print(f"{MATCHING_FILE}: {MATCHING_FILE.stat().st_size / 1e6:.0f} MB | {CANDIDATE_FILE}: {CANDIDATE_FILE.stat().st_size / 1e6:.0f} MB")

CONFIG = {"features": FEATURES, "model_file": str(MODEL_FILE), "threshold": T_BEST, "country_thresholds": COUNTRY_THRESHOLDS,
          "one_owner": True, "candidates_per_s1": CANDIDATES_PER_S1, "use_extra_candidates": USE_EXTRA_CANDIDATES,
          "pred_parts": PRED_PARTS, "test": TEST_STATS, "buckets": str(BUCKET_DIR), "processed": str(PROCESSED_DIR),
          "minutes": round((time.time() - T0) / 60, 1)}
_ = (OUT_DIR / "model" / "predict_config.json").write_text(json.dumps(CONFIG, indent=1, default=str), encoding="utf-8")


## Check the files against the submission rules
The same rules as `validate_submission.py`, including its optional ID-existence check: header,
one row per test S1 (no missing, extra or duplicate rows), no duplicate IDs in a list, only
S2-/S3- IDs, every ID exists in the test S2/S3 files, and matches ⊆ candidates. Then the official
validator runs too, if it is attached as an input (it is stdlib-only).

In [ ]:
def check_file(path, header, required, valid_ids):
    """Rule violations of one submission file, and its {s1: set(ids)} mapping."""
    problems, mapping, rows = [], {}, []
    with open(path, encoding="utf-8") as f:
        if f.readline().rstrip("\n").split("\t") != header:
            problems.append(f"header is not {header}")
        for line in f:
            s1, tab, rest = line.partition("\t")
            if not tab:
                problems.append(f"row without a tab: {line[:60]!r}")
                continue
            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            rows.append(s1)
            if len(ids) != len(set(ids)):
                problems.append(f"duplicate ID in the list of {s1}")
            mapping[s1] = set(ids)
    bad_prefix = {i for v in mapping.values() for i in v if not i.startswith(("S2-", "S3-"))}
    unknown = {i for v in mapping.values() for i in v if i not in valid_ids}
    for label, found in [("duplicate S1 rows", len(rows) - len(set(rows))), ("missing test S1", len(required - set(rows))),
                         ("S1 not in the test set", len(set(rows) - required)), ("IDs without S2-/S3- prefix", len(bad_prefix)),
                         ("IDs not in test S2/S3", len(unknown))]:
        if found:
            problems.append(f"{found:,} {label}")
    return problems, mapping


required = set(TEST_S1)
valid_ids = set(read_ids(TEST_DIR / "test_source2.tsv")) | set(read_ids(TEST_DIR / "test_source3.tsv"))
problems_m, matched = check_file(MATCHING_FILE, ["source1_entity_id", "matched_entity_ids"], required, valid_ids)
problems_c, candidates = check_file(CANDIDATE_FILE, ["source1_entity_id", "candidate_entity_ids"], required, valid_ids)
not_subset = sum(1 for s, v in matched.items() if v - candidates.get(s, set()))
print("matching_results.tsv:", problems_m or "OK")
print("candidate_pairs.tsv:", problems_c or "OK")
print("matches that are not candidates:", not_subset or "none")
assert not problems_m and not problems_c and not not_subset, "fix the problems above before submitting"
del valid_ids

if VALIDATOR is not None:
    result = subprocess.run([sys.executable, str(VALIDATOR), "--matching", str(MATCHING_FILE), "--candidate",
                             str(CANDIDATE_FILE), "--test-dir", str(TEST_DIR), "--check-ids"],
                            capture_output=True, text=True, encoding="utf-8", errors="replace")
    print(result.stdout[-4000:], result.stderr[-2000:])
else:
    print("validate_submission.py not attached: run it locally on the downloaded files (see the notebook text)")
print(f"done in {(time.time() - T0) / 60:.1f} min")